# 7. Prepare inputs for reaction-method development
**Input:** your cluster.extxyz produced in notebook 03.
**Output:** isolated pair, frozen-shell constraint and row-identified source-force archive.
**Modules:** ASE + geometry + visualization. No cmcc-reactions files are changed and no optimization runs.

The external-force and explicit-environment routes remain proposed research workflows. MartiniFuncs does not yet implement a validated candidate-to-TS pipeline. This preparation does not calculate a barrier.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from ase.io import read, write
from ase.constraints import FixAtoms
from MartiniFuncs.structure.alignment import align_geometry
from MartiniFuncs.chemistry.cpd_n4t.visualization import plot_cluster

CLUSTER_FILE = Path(r"/path/to/analysis/cluster.extxyz")
REFERENCE_FILE = None  # optional finite cluster with identical persistent IDs
OUTPUT_DIR = CLUSTER_FILE.parent / "reaction_preparation"
WRITE_FILES = False


### Mathematics: frozen environment and source-force archive
`pair_rows` and `shell_rows` select the recorded pair/shell roles. The exported array is exactly $\mathbf F_{\rm pair}=\mathbf F_{\rm MD}[\mathrm{pair\_rows}]$, with persistent IDs preserved. Freezing a shell sets
$$\mathbf r_i=\mathbf r_i^{(0)}\quad(i\in\mathrm{shell})$$
in later constrained optimization; this notebook only attaches `FixAtoms` and exports inputs. It does not compute an optimized structure, a transition state, or a barrier $\Delta E^\ddagger$.

Source MD forces are total forces. An environmental residual would require a separately justified subtraction such as $\mathbf F_{\rm env}=\mathbf F_{\rm full}-\mathbf F_{\rm isolated\ pair}$ at matched geometry and settings. **That subtraction is not performed here.** Adding total source forces to a pair calculator can double-count interactions. `REFERENCE_FILE` only invokes the rigid alignment described below, consistently rotating archived vectors.


### Mathematics: alignment and visualization units
Alignment matches persistent IDs and minimizes the equal-weight objective
$$\mathrm{RMSD}=\sqrt{\frac1N\sum_i\|\mathbf r_iQ+\mathbf t-\mathbf r_i^{\rm ref}\|^2}.$$
Centered coordinates give $X^TY=U\Sigma V^T$, $Q=U\,\mathrm{diag}(1,1,\det(UV^T))V^T$, $\mathbf t=\bar{\mathbf r}^{\rm ref}-\bar{\mathbf r}Q$. Coordinates transform as $\mathbf r'=\mathbf rQ+\mathbf t$; vectors as $\mathbf F'=\mathbf FQ$, $\mathbf v'=\mathbf vQ$ (no translation). RMSD is Å. A supplied reference changes orientation, not physical geometry.

Psience coordinates are Bohr: $\mathbf r_{\rm Bohr}=\mathbf r_{\mathrm{Angstrom}}/a_0$, $a_0\approx0.529177$ Å. The ASE-to-Psience conversion performs this step.


In [ ]:
cluster = read(CLUSTER_FILE)
for key in ("pair_role","global_index","lammps_forces"):
    if key not in cluster.arrays:
        raise ValueError(f"Missing extraction metadata: {key}")
if cluster.pbc.any():
    raise ValueError("Use a finite, whole extracted cluster")
if REFERENCE_FILE is not None:
    cluster, rmsd, transform = align_geometry(cluster, read(REFERENCE_FILE))
    print("Aligned RMSD (angstrom):",rmsd)
shell_rows = np.flatnonzero(cluster.arrays["pair_role"]=="shell")
pair_rows = np.flatnonzero(cluster.arrays["pair_role"]!="shell")
pair = cluster[pair_rows]
source_pair_forces = cluster.arrays["lammps_forces"][pair_rows].copy()
environment = cluster.copy()
environment.set_constraint(FixAtoms(indices=shell_rows))
display(pd.DataFrame({"pair_row":np.arange(len(pair)), "cluster_row":pair_rows,
                      "global_index":pair.arrays["global_index"],
                      "fx":source_pair_forces[:,0],"fy":source_pair_forces[:,1],"fz":source_pair_forces[:,2]}))
print("Frozen shell atoms:",len(shell_rows))
display(plot_cluster(environment))


## External-force route: unresolved physical model
Source MD forces are total periodic-system forces. Adding them unchanged to a pair MLIP calculation can double-count interactions. One possible quantity to investigate is full-frame pair force minus isolated-pair force at identical geometry; its meaning for a nonadditive MLIP, electronic state, truncation and changing geometry requires validation.

The inspected cmcc-reactions interface includes `ForceOptimizer.reoptimize_with_force` with mode, magnitude, units and projection/orientation options. It is not established here as a drop-in receiver of an N-by-3 source-force array. An adapter must verify force-versus-gradient sign, normalization, row ordering, coordinate frame, units and energy/work conventions.

## Explicit-environment route
`environment` carries an ASE FixAtoms constraint on the shell. Verify that the downstream optimizer honors it. Shell-radius convergence, finite boundaries, electronic state and UMA boundary/task support remain validation questions. No MD source-force array is automatically added to this model.

A barrier still needs defined endpoints/TS and a consistent energy convention. Compare both routes against controlled UMA reference calculations before interpreting barriers.

In [ ]:
if WRITE_FILES:
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    targets = [OUTPUT_DIR/name for name in ("pair.extxyz","frozen_shell.extxyz","source_forces.npz","preparation.json")]
    if any(path.exists() for path in targets):
        raise FileExistsError("Use a fresh reaction-preparation directory")
    write(targets[0],pair)
    write(targets[1],environment)
    np.savez(targets[2], forces_eV_angstrom=source_pair_forces,
             pair_global_ids=pair.arrays["global_index"],pair_cluster_rows=pair_rows,
             frozen_shell_rows=shell_rows)
    targets[3].write_text(json.dumps({"source_cluster":str(CLUSTER_FILE),
        "reference":str(REFERENCE_FILE) if REFERENCE_FILE is not None else None,
        "force_kind":"total source MD forces; not validated external forces",
        "force_units":"eV/Angstrom","position_units":"Angstrom",
        "frozen_shell_rows":shell_rows.tolist(),"barrier_calculated":False},indent=2)+chr(10))
    print("Saved:",OUTPUT_DIR)
